### Introduction
Col-0 possesses the ACQOS gene (At5g46520, an NLR also called VICTR) plus a neighboring NLR (At5g46510) within the RPS6 cluster
researchgate.net
. In contrast, certain accessions carry a deletion haplotype that lacks ACQOS (sequence present in Col-0 but deleted in these lines)

| Chr | Start   | End    |
|-----|---------|--------|
| 5   | 18867616  | 18872494 |

In [1]:
import pysam
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
import numpy as np

In [2]:
def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForMaskedLM.from_pretrained(model_name, trust_remote_code=True).eval()
    return tokenizer, model

In [3]:
genome_file = '../../../results/SV_effect/Arabidopsis_thaliana.TAIR10.dna.toplevel.noMtPt.fa'
fasta = pysam.FastaFile(genome_file)

In [4]:
start = 18867616
end = 18872495
chrom = '5'
del_len = end - start + 1
del_len

4880

In [5]:
ref_seq = fasta.fetch(chrom, start - 1 - 4096, end + 4096)
mut_seq = ref_seq[0:4096] + ref_seq[-4096:]
ref_seq = ref_seq[(del_len//2):-(del_len//2)]

In [6]:
len(ref_seq), len(mut_seq)

(8192, 8192)

In [7]:
tokenizer, model = load_model('kuleshov-group/compo-cad2-l48-d1536-dna-chtk-c8k-1t-v1-b2-lr4e4-NzqiLr')
model.to('cuda:0')

/home/jz963/miniconda3/envs/transformers/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


CaduceusForMaskedLM(
  (caduceus): Caduceus(
    (backbone): CaduceusMixerModel(
      (embeddings): CaduceusEmbeddings(
        (word_embeddings): RCPSEmbedding(
          (embedding): Embedding(8, 1536)
        )
      )
      (layers): ModuleList(
        (0-47): 48 x RCPSMambaBlock(
          (mixer): RCPSWrapper(
            (submodule): BiMambaWrapper(
              (mamba_fwd): Mamba2(
                (in_proj): Linear(in_features=1536, out_features=6320, bias=False)
                (conv1d): Conv1d(3200, 3200, kernel_size=(4,), stride=(1,), padding=(3,), groups=3200)
                (act): SiLU()
                (norm): RMSNorm()
                (out_proj): Linear(in_features=3072, out_features=1536, bias=False)
              )
              (mamba_rev): Mamba2(
                (in_proj): Linear(in_features=1536, out_features=6320, bias=False)
                (conv1d): Conv1d(3200, 3200, kernel_size=(4,), stride=(1,), padding=(3,), groups=3200)
                (act): SiLU()
   

In [8]:
inputs = tokenizer(
            [ref_seq, mut_seq],
            truncation=False,
            padding=False,
            return_tensors="pt",
            return_attention_mask=False,
            return_token_type_ids=False,)['input_ids'].to(model.device)

In [9]:
inputs.shape

torch.Size([2, 8192])

In [10]:
with torch.no_grad():
    outputs = model(inputs)

In [11]:
nucleotides = list('acgt')
logits = outputs.logits[..., [tokenizer.get_vocab()[nc] for nc in nucleotides]]
probs = torch.nn.functional.softmax(logits, dim=2).cpu().numpy()
probs.shape

(2, 8192, 4)

In [12]:
probs[0][0]

array([0.04673369, 0.03155079, 0.01655972, 0.9051558 ], dtype=float32)

In [13]:
flank_len = (8192-del_len)//2

In [14]:
ref_prob_left = probs[0][0:flank_len][-20:]
mut_prob_left = probs[1][0:4096][-20:]

ref_prob_right = probs[0][-flank_len:][0:20]
mut_prob_right = probs[1][-4096:][0:20]

In [15]:
ref_prob = np.concatenate((ref_prob_left, ref_prob_right), axis = 0)
mut_prob = np.concatenate((mut_prob_left, mut_prob_right), axis = 0)
seq = ref_seq[0:flank_len][-20:] + ref_seq[-flank_len:][0:20]

In [16]:
ref_prob.shape, mut_prob.shape, len(seq)

((40, 4), (40, 4), 40)

In [17]:
scores = []
nucleotides = "ACGT"
for idx, nt in enumerate(seq):
    if nt in nucleotides:
        refProb = ref_prob[idx, nucleotides.index(nt)]
        mutProb = mut_prob[idx, nucleotides.index(nt)]
        scores.append(np.log(mutProb / refProb))
    else:
        scores.append(0)

In [18]:
np.mean(scores)

-0.041588977

## The quantile of simulated deletions  using flanking 20bp (+10/-10)

| Percentile | Value      |
|------------|------------|
| 0.1%       | -1.6905446 |
| 1%         | -1.1289620 |
| 10%        | -0.4855522 |
| 50%        | -0.0777035 |

## Conclusion
Unfortunately, this insertion doesn't fall into the top 10%